# 02. Gameplay Summary & Cleaning

**Goal**: Analyze player activity, filter out short sessions, and trim long sessions.

**Inputs**:
- `data/processed/merged_telemetry.csv` (Output from Step 1)

**Outputs**:
- `data/processed/2_cleaned_telemetry_for_modelling.csv` (Filtered content)
- `data/processed/2_player_summary.csv` (Stats)

## 1. Setup and Imports

In [7]:
import pandas as pd
import os

# Inputs & Outputs
INPUT_PATH = 'data/processed/merged_telemetry.csv'
OUT_CLEANED = 'data/processed/2_cleaned_telemetry_for_modelling.csv'
OUT_SUMMARY = 'data/processed/2_player_summary.csv'

print("Libraries imported and paths defined.")
print(f"Input Path: {INPUT_PATH}")

Libraries imported and paths defined.
Input Path: data/processed/merged_telemetry.csv


## 2. Load Data

In [8]:
print("Loading processed telemetry data...")
if os.path.exists(INPUT_PATH):
    df = pd.read_csv(INPUT_PATH)
    print(f"Loaded {len(df)} rows. Unique Users: {df['userId'].nunique()}")
else:
    raise FileNotFoundError(f"File not found at {INPUT_PATH}. Please run 01_Data_Loading_and_Merging.ipynb first.")

Loading processed telemetry data...
Loaded 3492 rows. Unique Users: 58


## 3. Calculate Player Duration
We calculate the total duration played by each user based on the number of telemetry rows (assuming 30s intervals).

In [9]:
if 'userId' in df.columns:
    # Group keys: userId
    user_stats = df.groupby('userId').size().reset_index(name='total_rows')
    user_stats['duration_min'] = (user_stats['total_rows'] * 30) / 60
    
    print("\n--- Initial Duration Stats ---")
    print(user_stats['duration_min'].describe())
else:
    raise ValueError("'userId' column missing in dataset.")


--- Initial Duration Stats ---
count     58.000000
mean      30.103448
std       23.245780
min        1.000000
25%       12.750000
50%       31.250000
75%       38.500000
max      136.000000
Name: duration_min, dtype: float64


## 4. Filter Users (Minimum Duration)
We filter out users who played for less than 20 minutes.

In [10]:
min_duration = 20

valid_users = user_stats[user_stats['duration_min'] >= min_duration]
dropped_users = user_stats[user_stats['duration_min'] < min_duration]

print(f"Filtering Rule: Keep users with >= {min_duration} minutes of play.")
print(f"Keeping: {len(valid_users)} users.")
print(f"Dropping: {len(dropped_users)} users.")

Filtering Rule: Keep users with >= 20 minutes of play.
Keeping: 40 users.
Dropping: 18 users.


## 5. Trim Sessions (Maximum Duration)
We cap the analysis to the first 45 minutes of gameplay to ensure comparability.

In [11]:
cap_duration = 45
max_rows = int((cap_duration * 60) / 30) # 90 rows

print("Applying filtering and trimming...")
# We only keep rows for valid users
df_filtered = df[df['userId'].isin(valid_users['userId'])].copy()

# Sort properly to ensure we keep the FIRST 45 mins
t_col = 'timestamp' if 'timestamp' in df_filtered.columns else 'time'
print(f"Sorting by column: {t_col}")

# Ensure sorting column exists and is usable
if t_col in df_filtered.columns:
     # Helper for sort if not already done
     df_filtered['sort_helper'] = pd.to_numeric(df_filtered[t_col], errors='coerce')
     # If numeric conversion failed mostly (e.g. ISO strings), use datetime
     if df_filtered['sort_helper'].isna().sum() > len(df_filtered) * 0.5:
          df_filtered['sort_helper'] = pd.to_datetime(df_filtered[t_col], errors='coerce')
          
     df_filtered.sort_values(by=['userId', 'sort_helper'], inplace=True)
     df_filtered.drop(columns=['sort_helper'], inplace=True)

# Trim > 45 mins (Keep first 90 rows per user)
df_final = df_filtered.groupby('userId').head(max_rows)

print(f"Trimming: Reduced rows from {len(df_filtered)} to {len(df_final)} (Cap at {cap_duration}m / {max_rows} rows).")

Applying filtering and trimming...
Sorting by column: timestamp
Trimming: Reduced rows from 3253 to 2838 (Cap at 45m / 90 rows).


## 6. Final Summary and Export

In [12]:
# Generate stats for trimmed data
final_stats = df_final.groupby('userId').size().reset_index(name='Data points')
final_stats['Duration (minutes)'] = (final_stats['Data points'] * 30) / 60
final_stats.rename(columns={'userId': 'Player ID'}, inplace=True)
final_stats = final_stats[['Player ID', 'Duration (minutes)', 'Data points']]
final_stats.sort_values('Duration (minutes)', ascending=False, inplace=True)

print("\n--- Final Cleaned Summary (Top 5) ---")
print(final_stats.head(5).to_string(index=False))

# Export
os.makedirs('data/processed', exist_ok=True)
df_final.to_csv(OUT_CLEANED, index=False)
final_stats.to_csv(OUT_SUMMARY, index=False)

print(f"\nSaved cleaned dataset to: {OUT_CLEANED}")
print(f"Saved summary to: {OUT_SUMMARY}")
print("Gameplay summary processing completed.")


--- Final Cleaned Summary (Top 5) ---
               Player ID  Duration (minutes)  Data points
6974892348d53c4152cf1421                45.0           90
6974acd8315a39e91bc1e52f                45.0           90
6974f3cc5d8e98b89f0a5d07                45.0           90
697502455d8e98b89f0a9f38                45.0           90
6975de91547a2b541d1316de                45.0           90

Saved cleaned dataset to: data/processed/2_cleaned_telemetry_for_modelling.csv
Saved summary to: data/processed/2_player_summary.csv
Gameplay summary processing completed.
